
# Projeto: Funções Weather Webscraping

### Objetivo  
Este notebook implementa funções auxiliares para coletar, extrair e estruturar dados meteorológicos de forma padronizada do site Weather. Ele permite gerar DataFrames de previsão para múltiplas cidades brasileiras, com foco em integração com pipelines de dados.

### Informações Técnicas

- **Tecnologias Utilizadas**: 
  - Apache Spark (PySpark)
  - Databricks Notebooks
 
---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                |
|------------|------------------|--------|--------------------------------------------|
| 14/05/2025 | Lucas Berta e Evelise Dias      |  1.0   | Criação de funções auxiliares para coleta de dados|
| 16/05/2025 | Lucas Berta     |  1.1   | Ajuste de docstrings |
| 19/05/2025 | Evelise Dias   |  1.2   | Inclusão de documentação|


# Configurações Iniciais

In [0]:
%run ./funcoes_config

In [0]:
%run ./funcoes_comuns_motores

In [0]:
%run ../../../src/utils/utils_run 

In [0]:
import builtins


### Helpers (Funções Auxiliares)
Agrupamos aqui funções pequenas e específicas que isolam tarefas comuns e repetitivas, como manipulação de datas, construção de URLs e extração padronizada dos dados da API. Isso torna o código mais legível, evita duplicação e facilita ajustes futuros, sem misturar lógica de negócio com detalhes técnicos da API.

In [0]:
def get_html_from_url(cidade_id: str) -> str:
    """Obtém o HTML da página de previsão do tempo de uma cidade.

    Args:
        cidade_id (str): Identificador da cidade na URL da Weather.com.

    Returns:
        str: Código HTML da página.

    Raises:
        Exception: Caso a requisição falhe.
    """
    url = f"https://weather.com/pt-BR/clima/10dias/l/{cidade_id}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    raise Exception(f"Erro ao acessar a página, status_code: {response.status_code}")

In [0]:
def parse_forecast_data(html: str, data_atual: str) -> list[dict]:
    """Extrai informações de previsão do tempo do HTML.

    Args:
        html (str): Código HTML da página.
        data_atual (str): Data da coleta no formato "YYYY-MM-DD".

    Returns:
        list[dict]: Lista de previsões extraídas.
    """
    soup = BeautifulSoup(html, "html.parser")
    daily_elements = soup.select("[data-testid='DailyContent']")
    
    # Fuso horário de São Paulo
    tz_sp = pytz.timezone("America/Sao_Paulo") 
    agora_sp = datetime.now(tz_sp)
    hora_coleta = agora_sp.hour
    data_atual = agora_sp.strftime("%Y-%m-%d")

    records = []
    dia_index = 0

    # Verifica qual o periodo do dia para realizar as coletas
    for element in daily_elements:
        texto = element.get_text(strip=True)
        period = re.search(r'\|\s*(Dia|Noite)', texto)
        if not period:
            continue
        period_str = period.group(1)
        day = addDays(data_atual, dia_index)
        if period_str == "Dia" and dia_index == 0 and hora_coleta >= 17:
            continue
        
        # Coleta tempeaturas de probabilidade de chuva
        max_temp = re.search(r'Máxima de\s+(\d+°)', texto)
        min_temp = re.search(r'Mínima de\s+(\d+°)', texto)
        rain = re.search(r'Rain drop(\d+)%', texto)

        # Coleta informações referente ao percentual de umidade local
        umidade_el = element.select_one("[data-testid='PercentageValue']")
        umidade = umidade_el.get_text(strip=True) if umidade_el else ""

        # Coleta informações referente a direção e velocidade do vento no local
        wind_element = element.select_one("[data-testid='Wind']")
        if wind_element:
            spans = wind_element.find_all("span")
            direcao = spans[0].get_text(strip=True) if len(spans) > 0 else ""
            velocidade = spans[1].get_text(strip=True).replace("km/h", "").strip() if len(spans) > 1 else ""
            try:
                velocidade_num = int(velocidade)
            except:
                velocidade_num = None
            vento = f"{direcao} {velocidade_num} km/h" if direcao and velocidade_num is not None else ""
        else:
            vento = ""

        # Grava todos os dados coletados
        record = {
            "day": day,
            "period": period_str,
            "temperatura_maxima": max_temp.group(1) if max_temp else "",
            "temperatura_minima": min_temp.group(1) if min_temp else "",
            "precipitacao": rain.group(1) + "%" if rain else "",
            "vento": vento,
            "umidade_maxima": umidade if period_str == "Noite" else "",
            "umidade_minima": umidade if period_str == "Dia" else ""
        }
        records.append(record)

        if period_str == "Noite":
            dia_index += 1
    return records

In [0]:
def aggregate_forecast(records: list[dict]) -> list[dict]:
    """Agrupa previsões diárias em um único registro consolidado por dia.

    Args:
        records (list[dict]): Lista de previsões coletadas.

    Returns:
        list[dict]: Lista agregada de previsões organizadas por dia.
    """
    grouped = {}
    for r in records:
        key = r["day"]
        if key not in grouped:
            grouped[key] = []
        grouped[key].append(r)

    aggregated = []
    for day, items in grouped.items():
        maxs = [int(r["temperatura_maxima"].replace("°", "")) for r in items if r["temperatura_maxima"]]
        mins = [int(r["temperatura_minima"].replace("°", "")) for r in items if r["temperatura_minima"]]
        precip = next((r["precipitacao"] for r in items if r["period"] == "Dia" and r["precipitacao"]), "")
        wind = next((r["vento"] for r in items if r["vento"]), "")
        umidade_max = [int(r["umidade_maxima"].replace("%", "")) for r in items if r["umidade_maxima"]]
        umidade_min = [int(r["umidade_minima"].replace("%", "")) for r in items if r["umidade_minima"]]

        aggregated.append({
            "data_previsao": day,
            "temperatura_maxima": f"{builtins.max(maxs)}°" if maxs else "",
            "temperatura_minima": f"{builtins.min(mins)}°" if mins else "",
            "precipitacao": precip,
            "vento": wind,
            "umidade_maxima": f"{builtins.max(umidade_max)}%" if umidade_max else "",
            "umidade_minima": f"{builtins.min(umidade_min)}%" if umidade_min else "",
        })

    return aggregated

In [0]:
def generate_spark_df(data: list[dict]):
    """Cria um DataFrame do Spark a partir de uma lista de registros.

    Args:
        data (list[dict]): Lista de dados para conversão.

    Returns:
        DataFrame: DataFrame do Spark com os dados estruturados.
    """
    schema = StructType([
        StructField("data_previsao", StringType(), True),
        StructField("temperatura_maxima", StringType(), True),
        StructField("temperatura_minima", StringType(), True),
        StructField("precipitacao", StringType(), True),
        StructField("vento", StringType(), True),
        StructField("umidade_maxima", StringType(), True),
        StructField("umidade_minima", StringType(), True),
        StructField("estado", StringType(), True),
        StructField("cidade", StringType(), True),
    ])
    return spark.createDataFrame(data, schema=schema)


### Função principal que coleta previsões
Esta função orquestra a coleta das previsões para todas as capitais, usando os helpers para montar URLs e extrair dados, garantindo que cada etapa fique clara e modular. Ela centraliza a lógica do fluxo de dados, controle de erros e agrega informações locais para que o resultado seja uma lista pronta para análise.

In [0]:
def coletar_previsoes_refatorado(nome_enum: str):
    """Coleta previsões do tempo para diversas cidades e as converte em um DataFrame.

    Args:
        nome_enum (str): Nome da classe de enumeração contendo os IDs das cidades.

    Returns:
        DataFrame: DataFrame do Spark contendo previsões agregadas.
    """
    enum_class = getattr(enum_configs, nome_enum)
    all_data = []

    for uf, dados in enum_class.value.items():
        try:
            cidade_id = dados["id"]
            html = get_html_from_url(cidade_id)
            data_atual = datetime.now().strftime("%Y-%m-%d")
            registros = parse_forecast_data(html, data_atual)
            agregados = aggregate_forecast(registros)

            for a in agregados:
                a["estado"] = uf
                a["cidade"] = dados["capital"]

            all_data.extend(agregados)
        except Exception as e:
            print(f"Erro ao processar {uf} - {dados['capital']}: {e}")

    df = generate_spark_df(all_data)
    return df